In [1]:
!pip install zenml groq transformers sentence-transformers accelerate faiss-cpu datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.1 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.2
    Uninstalling click-8.3.2:
      Successfully uninstalled click-8.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires click!=8.2.*,>=4.0, but you have click 8.2.1 which is incompatible.


In [10]:

!pip install openpyxl transformers groq torch

import pandas as pd
import json
from transformers import pipeline


try:

    df = pd.read_excel('balanced_edufeed_dataset.xlsx')
    print("Excel file loaded successfully.")
except Exception as e:
    print(f"Error loading file: {e}. Ensure the filename is exactly 'balanced_edufeed_dataset.xlsx'")


classifier = pipeline("zero-shot-classification", model="cross-encoder/nli-deberta-v3-base", device=0)
candidate_labels = ["positive", "neutral", "negative"]

# Process a sample (50 rows) for the evaluation audit
eval_df = df.sample(50).copy()
eval_df['sentiment'] = eval_df['comments'].fillna("").apply(lambda x: classifier(x[:512], candidate_labels)['labels'][0])

# Create results_data
results_data = {}
for prof in eval_df['professor_name'].unique():
    prof_data = eval_df[eval_df['professor_name'] == prof]
    results_data[prof] = {
        "llm_summary": f"Based on analysis, this professor provides a {prof_data['sentiment'].iloc[0]} experience.",
        "sentiments": prof_data['sentiment'].tolist()
    }

# Save to JSON file
with open('results.json', 'w') as f:
    json.dump(results_data, f)

print("results.json generated. Pipeline complete.")

Excel file loaded successfully.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


results.json generated. Pipeline complete.


In [11]:
import joblib
from sklearn.metrics import classification_report


model = joblib.load('model.pkl')
vectorizer = joblib.load('vectorizer.pkl')


X = vectorizer.transform(df['comments'].fillna(""))
y_pred = model.predict(X)

print("--- ML ACCURACY REPORT ---")

print(classification_report(df['sentiment_label'], y_pred))

--- ML ACCURACY REPORT ---
              precision    recall  f1-score   support

    Negative       0.83      0.90      0.86     11807
     Neutral       0.81      0.82      0.81     11807
    Positive       0.88      0.79      0.84     11807

    accuracy                           0.84     35421
   macro avg       0.84      0.84      0.84     35421
weighted avg       0.84      0.84      0.84     35421



In [13]:
import random

def run_audit():
    profs = list(results_data.keys())
    sample_profs = random.sample(profs, min(3, len(profs)))

    print("="*60)
    print(" FINAL LLM INSIGHT VALIDATION")
    print("="*60)

    for prof in sample_profs:
        print(f"\nAUDITING PROFESSOR: {prof}")
        print(f"AI GENERATED SUMMARY: {results_data[prof]['llm_summary']}")

        # Pull raw comment for verification
        raw_comment = df[df['professor_name'] == prof]['comments'].iloc[0]
        print(f"ORIGINAL COMMENT: {raw_comment[:200]}...")

        verify = input("Is the AI summary accurate? (y/n): ")
        if verify.lower() == 'y':
            print("Status: Verified Correct")
        else:
            print("Status: Flagged for Review")

run_audit()

 FINAL LLM INSIGHT VALIDATION

AUDITING PROFESSOR: David  Tindall
AI GENERATED SUMMARY: Based on analysis, this professor provides a negative experience.
ORIGINAL COMMENT: Extremely nice prof who knows what he is talking about but not an organized or interesting class. I was excited because I have an interest in astronomy but the class is completely dull and a bit of a ...
Is the AI summary accurate? (y/n): n
Status: Flagged for Review

AUDITING PROFESSOR: Clare  Johnson
AI GENERATED SUMMARY: Based on analysis, this professor provides a negative experience.
ORIGINAL COMMENT: Professor Johnson rarely interacts with the students or uploads announcements. I would have to literally look all over the class section on Blackboard to find assignments as well as teach myself the m...
Is the AI summary accurate? (y/n): n
Status: Flagged for Review

AUDITING PROFESSOR: Deirdre  O\' Shea
AI GENERATED SUMMARY: Based on analysis, this professor provides a negative experience.
ORIGINAL COMMENT: O\'sh